# PY4Africa Day 6 — Plotly with IHS5 (Exercise)

Complete each TODO cell and add a short interpretation after each figure.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

In [ ]:
DATA_DIR = Path('../../data/0_raw/IHS 5 DATA sample')
print('Input folder:', DATA_DATA_DIRDIR)

### Load source tables

In [ ]:
hh = pd.read_stata(DATA_DIR / 'hh_mod_a_filt.dta', convert_categoricals=True)
roster = pd.read_stata(DATA_DIR / 'HH_MOD_B.dta', convert_categoricals=True)
edu = pd.read_stata(
    DATA_DIR / 'HH_MOD_C.dta',
    convert_categoricals=True,
    columns=['case_id', 'PID', 'hh_c08', 'hh_c09']
)
cons = pd.read_stata(DATA_DIR / 'ihs5_consumption_aggregate.dta', convert_categoricals=True)

print('Loaded shapes:')
print('  hh    ', hh.shape)
print('  roster', roster.shape)
print('  edu   ', edu.shape)
print('  cons  ', cons.shape)

## Build final analysis table (shared prep)

In [ ]:
# Household size and head profile
hh_size = roster.groupby('case_id').size().rename('hh_size')
is_head = (
    (pd.to_numeric(roster['hh_b04'], errors='coerce') == 1)
    | (roster['hh_b04'].astype('string').str.strip().str.upper() == 'HEAD')
)

head = roster.loc[is_head, ['case_id', 'PID', 'hh_b05a', 'hh_b03']].copy()
head.columns = ['case_id', 'head_pid', 'head_age', 'head_sex']
head['head_age'] = pd.to_numeric(head['head_age'], errors='coerce')

head_edu = edu[['case_id', 'PID', 'hh_c08', 'hh_c09']].copy()
head_edu.columns = ['case_id', 'head_pid', 'head_education_proxy', 'head_qualification_code']

cons['pcrexpagg'] = cons['rexpaggpc'].copy()
cons_sel = cons[['case_id', 'rexpagg', 'pcrexpagg', 'poor']].copy()

In [ ]:
df = (
    hh
    .merge(hh_size, on='case_id', how='left')
    .merge(head, on='case_id', how='left')
    .merge(head_edu, on=['case_id', 'head_pid'], how='left')
    .merge(cons_sel, on='case_id', how='left')
)

df['urban_rural'] = df['reside']
for col in ['hh_size', 'head_age', 'rexpagg', 'pcrexpagg']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f'Households: {len(df):,}')
df[['case_id', 'region', 'district', 'urban_rural', 'hh_size', 'pcrexpagg', 'poor']].head()

## 0) Histogram

In [ ]:
# TODO:
# Create a histogram of per-capita consumption using Plotly.
# - Use pcrexpagg clipped at the 99th percentile.
# - Color by urban_rural.
# - Overlay bars (not stacked).

hist_df = df[['pcrexpagg', 'urban_rural']].dropna().copy()
hist_df['pcrexpagg_clip'] = hist_df['pcrexpagg'].clip(upper=hist_df['pcrexpagg'].quantile(0.99))

# fig = px.histogram(...)
# fig.update_layout(...)
# fig.show()

## 1) Stacked bar chart

In [ ]:
# TODO:
# Build a stacked bar chart of household counts by urban_rural and poverty status (poor).

stack_df = (
    df[['urban_rural', 'poor']]
    .dropna()
    .groupby(['urban_rural', 'poor'], as_index=False)
    .size()
)

# fig = px.bar(..., barmode='stack')
# fig.show()

## 2) Grouped bar chart

In [ ]:
# TODO:
# Build a grouped bar chart of mean pcrexpagg by region and urban_rural.

group_df = (
    df[['region', 'urban_rural', 'pcrexpagg']]
    .dropna()
    .groupby(['region', 'urban_rural'], as_index=False)
    .agg(mean_pcrexpagg=('pcrexpagg', 'mean'))
)

# fig = px.bar(..., barmode='group')
# fig.show()

## 3) Timeline chart

In [ ]:
# TODO:
# Build a timeline (line chart) of monthly mean pcrexpagg by urban_rural.
# Use interviewDate to derive month.

time_df = df[['interviewDate', 'urban_rural', 'pcrexpagg']].dropna().copy()
time_df['interviewDate'] = pd.to_datetime(time_df['interviewDate'], errors='coerce')
time_df = time_df.dropna(subset=['interviewDate'])
time_df['month'] = time_df['interviewDate'].dt.to_period('M').dt.to_timestamp()

monthly = (
    time_df.groupby(['month', 'urban_rural'], as_index=False)
    .agg(mean_pcrexpagg=('pcrexpagg', 'mean'))
)

# fig = px.line(...)
# fig.show()

## 4) Scatter

In [ ]:
# TODO:
# Build a scatter plot of head_age vs pcrexpagg.
# - Color by urban_rural
# - Clip pcrexpagg at p99
# - Optionally sample rows for readability

scatter_df = df[['head_age', 'pcrexpagg', 'urban_rural', 'poor']].dropna().copy()
scatter_df = scatter_df[scatter_df['pcrexpagg'] <= scatter_df['pcrexpagg'].quantile(0.99)]
if len(scatter_df) > 4000:
    scatter_df = scatter_df.sample(4000, random_state=42)

# fig = px.scatter(...)
# fig.show()

## 4.1) Scatter + Regression Line

In [ ]:
# TODO:
# Add a regression line after the scatter chart.
# 1) Use head_age as x and pcrexpagg as y (clip y at p99).
# 2) Fit slope/intercept with np.polyfit.
# 3) Plot points + fitted line in one figure.

reg_df = df[['head_age', 'pcrexpagg', 'urban_rural']].dropna().copy()
reg_df = reg_df[reg_df['pcrexpagg'] <= reg_df['pcrexpagg'].quantile(0.99)]
if len(reg_df) > 4000:
    reg_df = reg_df.sample(4000, random_state=42)

# slope, intercept = np.polyfit(...)
# x_line = np.linspace(...)
# y_line = intercept + slope * x_line
# fig = px.scatter(...)
# fig.add_trace(go.Scatter(...))
# fig.show()

## 5) Bubble chart

In [ ]:
# TODO:
# Build a district-level bubble chart:
# - x: avg_head_age
# - y: avg_pcrexpagg
# - bubble size: n_households
# - color: region

bubble_df = (
    df[['district', 'region', 'head_age', 'pcrexpagg']]
    .dropna()
    .groupby(['district', 'region'], as_index=False)
    .agg(
        avg_head_age=('head_age', 'mean'),
        avg_pcrexpagg=('pcrexpagg', 'mean'),
        n_households=('head_age', 'size')
    )
    .sort_values('n_households', ascending=False)
    .head(25)
)

# fig = px.scatter(...)
# fig.show()

## 6) Sankey diagram (if possible)

In [ ]:
# TODO:
# Create a Sankey diagram from head_education_proxy -> poor.
# Tip:
# 1) Keep top education groups, collapse others to 'Other education'
# 2) Build a flow table with groupby(...).size()
# 3) Convert node names to source/target integer indices

sankey_df = df[['head_education_proxy', 'poor']].dropna().copy()
sankey_df['head_education_proxy'] = sankey_df['head_education_proxy'].astype('string').str.strip()
sankey_df['poor'] = sankey_df['poor'].astype('string').str.strip()

top_edu = sankey_df['head_education_proxy'].value_counts().head(8).index
sankey_df.loc[~sankey_df['head_education_proxy'].isin(top_edu), 'head_education_proxy'] = 'Other education'

flows = (
    sankey_df.groupby(['head_education_proxy', 'poor'], as_index=False)
    .size()
    .rename(columns={'size': 'value'})
)

# Build nodes + links and plot with go.Sankey
# fig = go.Figure(...)
# fig.show()

## 7) Custom Template

In [ ]:
# TODO:
# Create and register a custom Plotly template, then use it in at least one chart.

PALETTE = ["#7BB3B2", "#65A6BD", "#C997AF", "#B8B0D3", "#F4CF97", "#98B9A0", "#F6DECD"]
import plotly.graph_objects as go
import plotly.io as pio

nso_template = go.layout.Template()
nso_template.layout = go.Layout(
    font=dict(family='Arial', size=13, color='#333'),
    title_font=dict(size=16, color='#222'),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(showgrid=False),
    yaxis=dict(showgrid=True, gridcolor='lightgray', gridwidth=0.5),
    colorway=PALETTE,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=60, r=30, t=60, b=40)
)

# pio.templates['nso'] = nso_template
# fig = px.bar(..., template='nso')
# fig.show()

## 8) Pivot Tables + Pandas Formatting

In [ ]:
# TODO 8.1:
# Build a pivot table of mean pcrexpagg by region (rows) and urban_rural (columns),
# then format it with thousand separators and caption.

pivot_mean = df.pivot_table(
    values='pcrexpagg',
    index='region',
    columns='urban_rural',
    aggfunc='mean'
)

# pivot_mean.style.format('{:,.0f}').set_caption(
#     'Average Monthly Income by Region and Area Type (DKW)'
# )

In [ ]:
# TODO 8.2:
# Build a pivot table of poverty rate (share poor) by region and urban_rural,
# then format as percentages with a gradient.

tmp = df[['region', 'urban_rural', 'poor']].dropna().copy()
tmp['is_poor'] = (tmp['poor'].astype('string').str.strip().str.upper() == 'POOR').astype(float)

pivot_poor = tmp.pivot_table(
    values='is_poor',
    index='region',
    columns='urban_rural',
    aggfunc='mean'
)

# pivot_poor.style.format('{:.1%}').background_gradient(cmap='Reds').set_caption(
#     'Poverty Rate by Region and Area Type'
# )

## 9) Plotly Table Formatting

In [ ]:
# TODO:
# Render the pivot_mean table as a formatted Plotly table.
# - Keep region as first column.
# - Format values as comma-separated integers.

# table_df = pivot_mean.reset_index().copy()
# val_cols = [c for c in table_df.columns if c != 'region']
# fig = go.Figure(data=[go.Table(...)])
# fig.show()

## 10) Subplots

In [ ]:
# TODO:
# Create a 2x2 subplot dashboard using plotly.subplots.make_subplots.
# Suggested panels:
# 1) Histogram of pcrexpagg (clipped p99)
# 2) Bar chart: counts by urban_rural
# 3) Scatter: head_age vs pcrexpagg (sampled)
# 4) Box plot: pcrexpagg by urban_rural

from plotly.subplots import make_subplots

# fig = make_subplots(rows=2, cols=2, subplot_titles=[...])
# fig.add_trace(...)
# fig.update_layout(...)
# fig.show()

## Wrap-up
Write 2-3 lines comparing what each chart revealed that was not obvious from tables alone.